# Fase 2: Auditoría Adversaria: Ataques Avanzados

Autor: Daniel Gomollón Embid

Trabajo Fin de Grado: Análisis, Explotación y Mitigación de Vulnerabilidades de Sistemas de Detección de intrusiones basados en Machine Learning

Fecha: 30/03/2026

**Universidad de Zaragoza**

# ETAPA 3: Simulación de Ataques Reales: SPECTRA y DLA-Digital Twin

# **CORTEX — Transferibilidad de Ataques Adversariales (Black Box - No Brute Force)**

CORTEX: Collision-Oriented Representations for Twin Evasion eXploits

(Representaciones Orientadas a Colisiones para Exploits de Evasión Gemela)

## Hipótesis
Un atacante real no tiene acceso al modelo víctima (BigFlow-NIDS ResNet/LightGBM).
Solo dispone de un dataset público de tráfico de red (NF-UQ-NIDS-V2) con 44 features (finalmente 30 para adaptarnos a NetFlow puro y ser realistas ante el modelo víctima) - Sin IP Behavior Buffer ni features sintéticas.

¿Cuántos ataques adversariales generados sobre ese modelo sustituto (surrogate)
consiguen evadir al modelo víctima real?

## Métrica central: ITF (Inference Transfer Factor)
ITF = ataques que evaden víctima / ataques que evaden surrogate

------------------------

## Montar Drive y Configuraciones

In [ ]:
# 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

import sys
import os

# TODO: Cambia esta ruta a la carpeta raíz del TFG en Drive
PROJECT_ROOT = '/content/drive/MyDrive/Codigo_TFG'
sys.path.append(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Importar librerías core
import torch
import numpy as np
import polars as pl

import torch
import lightgbm as lgb
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import classification_report, f1_score

# Importar nuestros módulos
from src.config import Config
from src.helpers import set_seed
from src.utils.domain_constraints import DomainConstraints

# Fijar semilla para reproducibilidad
set_seed(Config.SEED)

# Comprobar hardware
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"[-] Ejecutando en dispositivo: {device.upper()}")

## 1. Carga del modelo víctima y datos de BigFlow-NIDS

El modelo víctima nunca se toca durante el entrenamiento del surrogate.
Solo se usa para evaluar cuántos ataques adversariales se transfieren.

In [ ]:
import joblib
import warnings
from src.models.wrapper_attacks import load_resnet_for_attack, load_lgbm_for_attack

# Silenciamos warnings inútiles
warnings.filterwarnings("ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

DATA_PATH   = os.path.join(PROJECT_ROOT, 'data', 'processed', 'resultados_2_buffer')
MODELS_PATH = os.path.join(PROJECT_ROOT, 'outputs', 'models')
SURROGATE_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'resultados_3_surrogate')

# Cargar datos de test escalados
print("[-] Cargando datos de test BigFlow-NIDS...")
X_test_victim_sc = np.load(os.path.join(DATA_PATH, 'X_test.npy')).astype(np.float32)
y_test_victim    = np.load(os.path.join(DATA_PATH, 'y_test.npy'))
print(f"   [✓] X_test escalado: {X_test_victim_sc.shape}")

# Cargar modelos víctima
print("\n[-] Cargando modelos víctima...")
resnet_victim  = load_resnet_for_attack(
    device=device,
    input_dim=X_test_victim_sc.shape[1],
    models_path=MODELS_PATH
)
lgbm_victim    = load_lgbm_for_attack(models_path=MODELS_PATH)
lgbm_explainer = joblib.load(os.path.join(MODELS_PATH, 'lgbm', 'lgbm_shap_explainer.pkl'))
print("   [✓] ResNet víctima cargada")
print("   [✓] LightGBM víctima cargado")

# Motor físico — necesario para invertir el escalado
dc = DomainConstraints.from_artifacts(models_path=MODELS_PATH, data_path=DATA_PATH)

# Invertir escalado para LightGBM (entrenado con datos raw)
print("\n[-] Invirtiendo escalado para LightGBM...")
X_test_victim_raw = dc.to_physical_space(X_test_victim_sc)
print(f"   [✓] X_test raw: {X_test_victim_raw.shape}")

# Submuestra de ataques — mismos índices para ambos espacios
np.random.seed(Config.SEED)
idx_attacks = np.where(y_test_victim != 0)[0]
np.random.shuffle(idx_attacks)
idx_attacks = idx_attacks[:2500]

X_victim_attacks_sc  = X_test_victim_sc[idx_attacks]   # para ResNet
X_victim_attacks_raw = X_test_victim_raw[idx_attacks]  # para LightGBM
y_victim_attacks     = y_test_victim[idx_attacks]

print(f"\n   [✓] Muestras de ataque: {len(idx_attacks):,}")

# Baseline — detección sin perturbación
y_pred_baseline_resnet = resnet_victim.predict(X_victim_attacks_sc)
y_pred_baseline_lgbm   = lgbm_victim.predict(X_victim_attacks_raw)

detection_resnet = (y_pred_baseline_resnet != 0).mean()
detection_lgbm   = (y_pred_baseline_lgbm   != 0).mean()

print(f"\n   Detección baseline ResNet  : {detection_resnet*100:.1f}%")
print(f"   Detección baseline LightGBM: {detection_lgbm*100:.1f}%")
print(f"\n   Ambos deberían estar ~95%+ para validar el experimento")

## 2. Carga y entrenamiento del modelo surrogate

El surrogate se entrena con NF-UQ-NIDS-V2 (44 features, sin IP Buffer).
El atacante nunca ha visto BigFlow-NIDS ni el modelo víctima.

Las features BUF_* se rellenan a cero al transferir los ataques al espacio
del modelo víctima — el atacante desconoce la existencia del buffer.

In [ ]:
import time
import lightgbm as lgb
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import seaborn as sns

SURROGATE_MODELS_PATH_BASE = os.path.join(PROJECT_ROOT, 'outputs', 'models', 'surrogate')
SURROGATE_MODELS_PATH_RESNET = os.path.join(SURROGATE_MODELS_PATH_BASE, 'resnet')
os.makedirs(SURROGATE_MODELS_PATH_RESNET, exist_ok=True)

print("[-] Cargando datos surrogate (NF-UQ-NIDS-V2, 30 features)...")
X_surr_train = np.load(os.path.join(SURROGATE_DATA_PATH, 'X_train.npy')).astype(np.float32)
y_surr_train = np.load(os.path.join(SURROGATE_DATA_PATH, 'y_train.npy'))
X_surr_val   = np.load(os.path.join(SURROGATE_DATA_PATH, 'X_val.npy')).astype(np.float32)
y_surr_val   = np.load(os.path.join(SURROGATE_DATA_PATH, 'y_val.npy'))
X_surr_test  = np.load(os.path.join(SURROGATE_DATA_PATH, 'X_test.npy')).astype(np.float32)
y_surr_test  = np.load(os.path.join(SURROGATE_DATA_PATH, 'y_test.npy'))

# Cargamos los feature names desde la carpeta base del surrogate
feature_names_surrogate = np.load(
    os.path.join(SURROGATE_MODELS_PATH_BASE, 'feature_names.npy'), allow_pickle=True
)

print(f"   [✓] Train: {X_surr_train.shape} | Val: {X_surr_val.shape} | Test: {X_surr_test.shape}")
print(f"   [✓] Features surrogate: {len(feature_names_surrogate)}")
print(f"   [✓] Features víctima  : {len(dc.feature_names)} (66 con BUF_*)")
print(f"   Features no disponibles para el atacante: "
      f"{len(dc.feature_names) - len(feature_names_surrogate)} (BUF_* + sintéticas)")

# Verificación de seguridad para gradientes
print(f"\n[-] Verificando escalado para redes neuronales...")
print(f"   [!] Rango X_surr_train: min={X_surr_train.min():.4f}, max={X_surr_train.max():.4f}")

In [ ]:
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, f1_score

print("\n[-] Entrenando Tabular ResNet Surrogate (Digital Twin)...")
t0 = time.time()

# 1. Definición de la Arquitectura (La navaja suiza del atacante)
class SurrogateResNet(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=128, num_blocks=3):
        super().__init__()
        self.entrada = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim)
            ) for _ in range(num_blocks)
        ])
        self.relu = nn.ReLU()
        self.salida = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.entrada(x)
        for block in self.blocks:
            x = self.relu(block(x) + x) # Skip connection: Vital para que DLA funcione
        return self.salida(x)

# 2. Preparar DataLoaders para PyTorch
BATCH_SIZE = 1024
num_classes = len(np.unique(y_surr_train))

train_loader = DataLoader(TensorDataset(torch.tensor(X_surr_train), torch.tensor(y_surr_train, dtype=torch.long)),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(torch.tensor(X_surr_val), torch.tensor(y_surr_val, dtype=torch.long)),
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(torch.tensor(X_surr_test), torch.tensor(y_surr_test, dtype=torch.long)),
                          batch_size=BATCH_SIZE, shuffle=False)

# 3. Inicialización
surrogate_nn = SurrogateResNet(input_dim=30, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(surrogate_nn.parameters(), lr=0.005, weight_decay=1e-4)

# 4. Bucle de Entrenamiento con Early Stopping
EPOCHS = 40
PATIENCE = 5
best_f1 = 0.0
best_wts = copy.deepcopy(surrogate_nn.state_dict())
patience_counter = 0

for epoch in range(EPOCHS):
    surrogate_nn.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(surrogate_nn(X_batch), y_batch)
        loss.backward()
        optimizer.step()

    # Validación
    surrogate_nn.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            preds = torch.argmax(surrogate_nn(X_batch.to(device)), dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(y_batch.numpy())

    val_f1 = f1_score(val_targets, val_preds, average='macro')

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_wts = copy.deepcopy(surrogate_nn.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"   [!] Early stopping en época {epoch+1} (Mejor F1 Val: {best_f1*100:.2f}%)")
            break

t_train = time.time() - t0
surrogate_nn.load_state_dict(best_wts) # Restaurar mejores pesos

# 5. Predicciones sobre Test
surrogate_nn.eval()
test_preds, test_targets = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = torch.argmax(surrogate_nn(X_batch.to(device)), dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_targets.extend(y_batch.numpy())

f1_surr = f1_score(test_targets, test_preds, average='macro')

print(f"\n   [✓] Entrenado en {t_train:.1f}s | Parámetros: {sum(p.numel() for p in surrogate_nn.parameters())}")
print(f"   [✓] F1-macro (test NF-UQ): {f1_surr*100:.2f}%\n")

# 6. Métricas por clase
CLASS_NAMES_SURR = Config.CLASS_NAMES
print(f"   Métricas por clase (test NF-UQ-NIDS-V2):")
print(f"   {'Clase':<20} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Soporte':>10}")
print(f"   {'─'*62}")

report = classification_report(test_targets, test_preds, target_names=CLASS_NAMES_SURR, output_dict=True, zero_division=0)
for cls_name in CLASS_NAMES_SURR:
    if cls_name in report:
        r = report[cls_name]
        print(f"   {cls_name:<20} {r['precision']:>10.3f} {r['recall']:>10.3f} {r['f1-score']:>10.3f} {int(r['support']):>10,}")

print(f"\n   Nota: Al igual que el LightGBM, la red neuronal clon aprende rápido los patrones")
print(f"         de NF-UQ. La clave será ver cómo transfiere sus gradientes a la ResNet víctima.")

# 7. Guardar el modelo en la subcarpeta resnet
torch.save(surrogate_nn.state_dict(), os.path.join(SURROGATE_MODELS_PATH_RESNET, 'surrogate_resnet.pth'))
print(f"\n   [✓] Digital Twin guardado en {SURROGATE_MODELS_PATH_RESNET}")

## 3. Construcción del espacio de ataque

El atacante genera ataques adversariales sobre el surrogate.
Para transferirlos al modelo víctima necesita expandir el vector
de 44 features a 66 features — las BUF_* se ponen a cero
(el atacante desconoce el IP Buffer del NIDS víctima).

## **DC ligero para el surrogate**

In [ ]:
"""
Domain Constraints ligero para el surrogate NF-UQ-NIDS-V2.
El atacante conoce las restricciones físicas del protocolo de red
aunque no conozca el modelo víctima ni su pipeline de features.
"""
from dataclasses import dataclass, field
import numpy as np
import joblib
from sklearn.preprocessing import QuantileTransformer

@dataclass
class SurrogateDomainConstraints:
    """
    Motor físico simplificado para el surrogate de 30 features.

    El atacante sabe que:
      - No puede generar bytes negativos
      - No puede poner puertos > 65535
      - Las features derivadas deben ser coherentes
      - El protocolo TCP tiene restricciones de tamaño de paquete

    No necesita saber nada del modelo víctima para esto.
    """
    feature_names  : np.ndarray
    scaler         : object          # QuantileTransformer del surrogate
    forward_mask   : np.ndarray = field(init=False)
    perturbable_mask: np.ndarray = field(init=False)

    # Las mismas categorías físicas que BigFlow — el protocolo no cambia
    FORWARD_SURR = [
        'IN_BYTES', 'IN_PKTS', 'MIN_TTL',
        'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT',
        'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
        'SRC_TO_DST_SECOND_BYTES', 'SRC_TO_DST_AVG_THROUGHPUT',
        'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES',
        'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES',
        'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN',
        'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX',
        'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV',
        'FLOW_DURATION_MILLISECONDS', 'DURATION_IN',
    ]

    # Límites físicos absolutos — mismos que BigFlow porque son del protocolo
    PHYSICAL_BOUNDS_SURR = {
        'IN_BYTES'                   : (40.0,    np.inf),
        'IN_PKTS'                    : (1.0,     140_225.0),
        'FLOW_DURATION_MILLISECONDS' : (0.0,     120_830.0),
        'MIN_TTL'                    : (0.0,     255.0),
        'LONGEST_FLOW_PKT'           : (28.0,    1500.0),
        'SHORTEST_FLOW_PKT'          : (28.0,    1500.0),
        'MIN_IP_PKT_LEN'             : (0.0,     1500.0),
        'MAX_IP_PKT_LEN'             : (28.0,    1500.0),
        'TCP_WIN_MAX_IN'             : (0.0,     65535.0),
        'SRC_TO_DST_IAT_MIN'         : (0.0,     59_995.0),
        'SRC_TO_DST_IAT_MAX'         : (0.0,     60_636.0),
        'SRC_TO_DST_IAT_AVG'         : (0.0,     59_996.0),
        'SRC_TO_DST_IAT_STDDEV'      : (0.0,     29_325.0),
        'SRC_TO_DST_AVG_THROUGHPUT'  : (6.0,     14_654_532.0),
        'NUM_PKTS_UP_TO_128_BYTES'   : (0.0,     140_225.0),
    }

    def __post_init__(self):
        n = len(self.feature_names)
        self.forward_mask     = self._build_mask(self.FORWARD_SURR, n)
        self.perturbable_mask = self.forward_mask.copy()

    def _build_mask(self, feature_list, n):
        mask = np.zeros(n, dtype=bool)
        for feat in feature_list:
            idx = self._feat_idx(feat)
            if idx is not None:
                mask[idx] = True
        return mask

    def _feat_idx(self, name):
        matches = np.where(self.feature_names == name)[0]
        return int(matches[0]) if len(matches) > 0 else None

    def to_physical_space(self, X_scaled):
        return self.scaler.inverse_transform(X_scaled).astype(np.float64)

    def to_scaled_space(self, X_raw):
        return self.scaler.transform(X_raw).astype(np.float32)

    def apply_causal_graph(self, X_phys):
        """
        Saneamiento causal sobre las 30 features del surrogate.
        Solo recalcula las derivadas disponibles en NF-UQ.
        """
        def safe_div(num, den):
            return num / (den + 1e-8)

        i_in_bytes  = self._feat_idx('IN_BYTES')
        i_in_pkts   = self._feat_idx('IN_PKTS')
        i_out_bytes = self._feat_idx('OUT_BYTES')
        i_out_pkts  = self._feat_idx('OUT_PKTS')
        i_dur       = self._feat_idx('FLOW_DURATION_MILLISECONDS')

        # Clipping físico absoluto
        for feat, (lo, hi) in self.PHYSICAL_BOUNDS_SURR.items():
            idx = self._feat_idx(feat)
            if idx is not None:
                X_phys[:, idx] = np.clip(X_phys[:, idx], lo, hi)

        # Recalcular derivadas disponibles
        idx = self._feat_idx('TOTAL_BYTES')
        if idx is not None and i_in_bytes is not None and i_out_bytes is not None:
            X_phys[:, idx] = X_phys[:, i_in_bytes] + X_phys[:, i_out_bytes]

        idx = self._feat_idx('SRC_BYTES_PER_PKT')
        if idx is not None and i_in_bytes is not None and i_in_pkts is not None:
            X_phys[:, idx] = safe_div(X_phys[:, i_in_bytes], X_phys[:, i_in_pkts])

        idx = self._feat_idx('PKT_RATIO')
        if idx is not None and i_in_pkts is not None and i_out_pkts is not None:
            X_phys[:, idx] = safe_div(X_phys[:, i_in_pkts], X_phys[:, i_out_pkts])

        idx = self._feat_idx('IS_UNIDIRECTIONAL')
        if idx is not None and i_out_pkts is not None:
            X_phys[:, idx] = np.where(X_phys[:, i_out_pkts] == 0, 1.0, 0.0)

        return X_phys

    def summary(self):
        print(f"SurrogateDC | Features: {len(self.feature_names)} | "
              f"Forward: {self.forward_mask.sum()} | "
              f"Frozen: {(~self.forward_mask).sum()}")

# Instanciar DC del surrogate (Ajuste de la ruta BASE)
surrogate_scaler = joblib.load(
    os.path.join(SURROGATE_MODELS_PATH_BASE, 'quantile_scaler.pkl')
)

dc_surrogate = SurrogateDomainConstraints(
    feature_names = feature_names_surrogate,
    scaler        = surrogate_scaler,
)
dc_surrogate.summary()

print(f"\n   Forward surrogate ({dc_surrogate.forward_mask.sum()} features):")
for i, name in enumerate(feature_names_surrogate):
    if dc_surrogate.forward_mask[i]:
        print(f"     [{i:>2}] {name}")

## **Wrapper del surrogate + mapeo de espacios**

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn

print("="*80)
print("CONSTRUYENDO CONEXIÓN PARA ATAQUES Y RESTRICCIONES FÍSICAS: Neural Wrapper & Causal Mapping")
print("="*80)

# Aseguramos que la red esté en modo evaluación
surrogate_nn.eval()

class NeuralSurrogateWrapper(nn.Module):
    """
    Wrapper compatible con ataques basados en gradientes (DLA, ACE).
    Al heredar de nn.Module, permite que las librerías de ataque calculen
    derivadas (loss.backward()) a través de este bloque.
    La red neuronal trabaja de forma nativa en espacio ESCALADO.
    """
    def __init__(self, nn_model):
        super().__init__()
        self.model = nn_model

    def forward(self, x_scaled_tensor):
        # Los ataques (DLA/ACE) pasarán tensores escalados por aquí
        return self.model(x_scaled_tensor)

    def predict(self, X_numpy_scaled):
        """Método de conveniencia para validaciones rápidas en Numpy"""
        self.model.eval()
        with torch.no_grad():
            tensor = torch.tensor(X_numpy_scaled, dtype=torch.float32).to(device)
            logits = self.model(tensor)
            return torch.argmax(logits, dim=1).cpu().numpy()

    def predict_proba(self, X_numpy_scaled):
        self.model.eval()
        with torch.no_grad():
            tensor = torch.tensor(X_numpy_scaled, dtype=torch.float32).to(device)
            logits = self.model(tensor)
            return torch.softmax(logits, dim=1).cpu().numpy()

# Instanciar el wrapper neuronal
surrogate_wrapper = NeuralSurrogateWrapper(surrogate_nn).to(device)

# ─── Mapeo de índices surrogate ↔ víctima ──────────────────────────────
print("[-] Construyendo mapeo bidireccional Surrogate (30) ↔ Víctima (66)...")

surrogate_to_victim = {}  # surr_idx → victim_idx
victim_to_surrogate = {}  # victim_idx → surr_idx

for surr_idx, feat in enumerate(feature_names_surrogate):
    victim_matches = np.where(dc.feature_names == feat)[0]
    if len(victim_matches) > 0:
        victim_idx = int(victim_matches[0])
        surrogate_to_victim[surr_idx] = victim_idx
        victim_to_surrogate[victim_idx] = surr_idx

n_mapeadas = len(surrogate_to_victim)
print(f"   [✓] Features mapeadas: {n_mapeadas}/{len(feature_names_surrogate)}")
print(f"   [✓] Features BUF_* (ciegas para el atacante): "
      f"{dc.immutable_mask.sum()} (se mantienen del original)")


# ─── Funciones de Proyección entre Mundos ──────────────────────────────
def project_victim_to_surrogate(X_victim_raw):
    """
    Proyecta flujos del espacio víctima (66 raw) al espacio surrogate (30 raw).
    """
    n = len(X_victim_raw)
    X_surr = np.zeros((n, len(feature_names_surrogate)), dtype=np.float32)
    for surr_idx, victim_idx in surrogate_to_victim.items():
        X_surr[:, surr_idx] = X_victim_raw[:, victim_idx]
    return X_surr


def expand_surrogate_to_victim(X_surr_adv_raw, X_victim_raw_original):
    """
    Expande adversariales del surrogate (30 raw) al espacio víctima (66 raw).
    Las features BUF_* y sintéticas mantienen su valor original inmutable.
    Aplica el grafo causal de la víctima (66) para mantener coherencia física.
    """
    X_expanded_raw = X_victim_raw_original.copy()

    # Inyectar las perturbaciones del atacante
    for surr_idx, victim_idx in surrogate_to_victim.items():
        X_expanded_raw[:, victim_idx] = X_surr_adv_raw[:, surr_idx]

    # Aplicar la física del IDS víctima
    X_expanded_raw = dc.apply_causal_graph(X_expanded_raw)

    # Re-escalar para la ResNet en producción
    X_expanded_sc = dc.to_scaled_space(X_expanded_raw)

    return X_expanded_raw, X_expanded_sc

# ─── Proyección de datos y validación de alineación ────────────────────
# Proyectar flujos víctima al espacio surrogate (Físico)
X_victim_in_surr_raw = project_victim_to_surrogate(X_victim_attacks_raw)
# Traducir al espacio latente/escalado del Digital Twin
X_victim_in_surr_sc  = dc_surrogate.to_scaled_space(X_victim_in_surr_raw)

print(f"\n   [✓] Flujos víctima en espacio surrogate (raw): {X_victim_in_surr_raw.shape}")
print(f"   [✓] Flujos víctima en espacio surrogate (sc) : {X_victim_in_surr_sc.shape}")

# Validar detección del Digital Twin (Atención: Predice sobre ESCALADO)
y_pred_surr_orig = surrogate_wrapper.predict(X_victim_in_surr_sc)
detection_surr   = (y_pred_surr_orig != 0).mean()

print(f"\n   Detección Surrogate sobre flujos BigFlow: {detection_surr*100:.1f}%")
print(f"   Detección Víctima LightGBM              : {detection_lgbm*100:.1f}%")
print(f"   Detección Víctima ResNet                : {detection_resnet*100:.1f}%")

# Solo atacar lo que el surrogate también detecta (Transferibilidad honesta)
mask_detected_by_surr = y_pred_surr_orig != 0
n_validas = mask_detected_by_surr.sum()
print(f"\n   Muestras válidas (detectadas por ambos): {n_validas:,}/{len(y_victim_attacks):,}")
print(f"   Estas son las que usaremos para nuestro ataque Neuronal.")

## **HIPÓTESIS:**
### Si se entrena un IDS basado en redes neuronales sin contexto temporal (IP Buffer) y en datasets artificiales, este modelo será extremadamente frágil a la hora de generalizar a redes en producción.

Reumen de resultados:

- Surrogate (30 feat, sin buffer) → 20.1% detección

- Víctima   (66 feat, con buffer) → 98%+  detección

Diferencia: +78 puntos porcentuales

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

modelos  = ['ResNet Surrogate\n(30 feat, sin buffer)',
            'LightGBM víctima\n(66 feat, con buffer)',
            'ResNet víctima\n(66 feat, con buffer)']
valores  = [detection_surr*100, detection_lgbm*100, detection_resnet*100]
colores  = ['#BA7517', '#E24B4A', '#1D9E75']

bars = ax.bar(modelos, valores, color=colores, width=0.5, edgecolor='none')
ax.set_ylabel('Tasa de detección de ataques (%)', fontweight='bold')
ax.set_title('Impacto del IP Behavior Buffer en la detección\n'
             '(mismo conjunto de 2500 flujos de ataque BigFlow-NIDS)',
             fontweight='bold', pad=12)
ax.set_ylim(0, 115)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Umbral 50%')

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)

ax.grid(axis='y', linestyle=':', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

# **SPECTRA: LEAF + THORN dual sobre surrogate**

In [ ]:
"""
CORTEX: Collision-Oriented Representations for Twin Evasion eXploits
(Representaciones Orientadas a Colisiones para Exploits de Evasión Gemela)

El atacante usa su propio modelo con su propio DC para generar
adversariales físicamente válidos en su espacio de 30 features.
Luego los expande al espacio de 66 features del víctima.
"""
import time
import warnings
from tqdm.notebook import tqdm

from src.attacks.ace import ACEAttack
from src.attacks.dla import DLAAttack

# --- SILENCIADOR DE WARNINGS ---
warnings.filterwarnings("ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'resultados_cortex')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# --- FUNCIÓN DE BARRAS DE PROGRESO ---
def ejecutar_con_barra(ataque, X, y, modelo, nombre_ataque, color_barra):
    ataque.verbose = False
    chunk_size = 100
    evasiones_totales = 0
    X_adv_lista = []
    ultimo_res = None

    for i in tqdm(range(0, len(X), chunk_size), desc=f"   {nombre_ataque}", leave=False, colour=color_barra):
        end = min(i + chunk_size, len(X))
        res_chunk = ataque.run(X[i:end], y[i:end], modelo)
        evasiones_totales += int(np.round(res_chunk.asr * len(X[i:end])))
        X_adv_lista.append(res_chunk.X_adv)
        ultimo_res = res_chunk

    asr_total = evasiones_totales / len(X)
    X_adv_total = np.vstack(X_adv_lista)
    ultimo_res.asr = asr_total
    ultimo_res.X_adv = X_adv_total

    return asr_total, ultimo_res

print("="*100)
print(f"[>>] SUITE OFENSIVA CORTEX: ACE vs DLA (Transferibilidad Grey-Box)")
print("="*100)
print(f"   Digital Twin: SurrogateResNet NF-UQ-NIDS-V2 (30 features)")
print(f"   Víctima     : LightGBM + ResNet BigFlow-NIDS (66 features)")

# 1. Filtramos: Solo atacar lo que el Digital Twin detecta (ITF honesto)
mask_detected_by_surr = y_pred_surr_orig != 0
n_validas = mask_detected_by_surr.sum()

# Subconjunto válido
X_surr_sc_valid  = X_victim_in_surr_sc[mask_detected_by_surr]
X_surr_raw_valid = X_victim_in_surr_raw[mask_detected_by_surr]
X_vic_raw_valid  = X_victim_attacks_raw[mask_detected_by_surr]
y_valid          = y_victim_attacks[mask_detected_by_surr]

print(f"\n[*] Muestras válidas (detectadas por el Digital Twin): {n_validas:,}/{len(y_victim_attacks):,}")

# 2. Extracción de "Anclas" Benignas para DLA (Deep Latent Anchoring)
print("[-] Extrayendo anclas benignas para forzar colisiones en DLA...")
idx_benign = np.where(y_test_victim == 0)[0]
np.random.seed(42)
np.random.shuffle(idx_benign)
idx_anchors = idx_benign[:50]

X_anchors_vic_raw = X_test_victim_raw[idx_anchors]
X_anchors_surr_raw = project_victim_to_surrogate(X_anchors_vic_raw)
X_anchors_surr_sc  = dc_surrogate.to_scaled_space(X_anchors_surr_raw)

# Capa objetivo del Digital Twin para interceptar la topología representacional
target_layer_dla = surrogate_nn.blocks[-1]

print(f"   [✓] {len(X_anchors_surr_sc)} anclas proyectadas al espacio latente.")
print("-" * 100)

epsilons = [0.05, 0.15, 0.30, 0.50, 1.0, 2.0]
resultados_cortex = []

for eps in epsilons:
    print(f"\nEvaluando Epsilon = {eps:.2f}")
    print("-" * 100)

    # ── 1. ACE OPTIMIZADO (Atacando al Gemelo) ──
    atk_ace = ACEAttack(
        constraints=dc_surrogate, epsilon=eps, alpha=eps/3.0, steps=10,
        momentum=0.0, random_start=True,
        adaptive_alpha=False,   # desactivado para que no frene el paso (mejora ataque, veamos transferibilidad)
        device=device
    )
    start_t = time.time()
    asr_ace_surr, res_ace = ejecutar_con_barra(atk_ace, X_surr_sc_valid, y_valid, surrogate_wrapper, f"ACE ε={eps}", "dodgerblue")
    t_ace = time.time() - start_t

    # Expansión y Evaluación de Transferibilidad (ACE)
    X_ace_adv_raw = dc_surrogate.to_physical_space(res_ace.X_adv)
    X_ace_vic_raw, X_ace_vic_sc = expand_surrogate_to_victim(X_ace_adv_raw, X_vic_raw_valid)

    asr_ace_lgbm   = (lgbm_victim.predict(X_ace_vic_raw) == 0).mean()
    asr_ace_resnet = (resnet_victim.predict(X_ace_vic_sc) == 0).mean()
    itf_ace_resnet = asr_ace_resnet / asr_ace_surr if asr_ace_surr > 0.005 else 0.0

    resultados_cortex.append({
        "Epsilon": eps, "Ataque": "ACE",
        "ASR Surrogate (%)": asr_ace_surr * 100,
        "ASR Vic LightGBM (%)": asr_ace_lgbm * 100,
        "ASR Vic ResNet (%)": asr_ace_resnet * 100,
        "ITF ResNet": itf_ace_resnet,
        "Tiempo": t_ace
    })

    print(f"  ↳ [ACE] Surr: {asr_ace_surr*100:5.1f}% | Vic ResNet: {asr_ace_resnet*100:5.1f}% | ITF: {itf_ace_resnet:.3f}")

    # ── 2. DLA GREY-BOX (CORTEX - Colisiones Latentes en el Gemelo) ──
    atk_dla = DLAAttack(
        constraints=dc_surrogate, target_layer=target_layer_dla, X_anchors=X_anchors_surr_sc,
        extraction_mode='grey_box', epsilon=eps, alpha=eps/3.0, steps=10,
        latent_weight=0.2, adaptive_alpha=True, device=device
    )
    start_t = time.time()
    asr_dla_surr, res_dla = ejecutar_con_barra(atk_dla, X_surr_sc_valid, y_valid, surrogate_wrapper, f"DLA ε={eps}", "mediumseagreen")
    t_dla = time.time() - start_t

    # Expansión y Evaluación de Transferibilidad (DLA)
    X_dla_adv_raw = dc_surrogate.to_physical_space(res_dla.X_adv)
    X_dla_vic_raw, X_dla_vic_sc = expand_surrogate_to_victim(X_dla_adv_raw, X_vic_raw_valid)

    asr_dla_lgbm   = (lgbm_victim.predict(X_dla_vic_raw) == 0).mean()
    asr_dla_resnet = (resnet_victim.predict(X_dla_vic_sc) == 0).mean()
    itf_dla_resnet = asr_dla_resnet / asr_dla_surr if asr_dla_surr > 0.005 else 0.0

    resultados_cortex.append({
        "Epsilon": eps, "Ataque": "DLA (CORTEX)",
        "ASR Surrogate (%)": asr_dla_surr * 100,
        "ASR Vic LightGBM (%)": asr_dla_lgbm * 100,
        "ASR Vic ResNet (%)": asr_dla_resnet * 100,
        "ITF ResNet": itf_dla_resnet,
        "Tiempo": t_dla
    })

    print(f"  ↳ [DLA] Surr: {asr_dla_surr*100:5.1f}% | Vic ResNet: {asr_dla_resnet*100:5.1f}% | ITF: {itf_dla_resnet:.3f}")

# --- GUARDADO Y VISUALIZACIÓN DEFINITIVA ---
print("\n\nTABLA DE RESULTADOS FINAL (CORTEX - CAJA GRIS):")

# 1. Creamos el DataFrame desde la memoria
df_raw = pd.DataFrame(resultados_cortex)

# 2. Renombramos las métricas para que el pivote sea limpio
df_raw = df_raw.rename(columns={
    "ASR Surrogate (%)": "ASR_Surr",
    "ASR Vic LightGBM (%)": "ITF_LGBM",
    "ASR Vic ResNet (%)": "ITF_ResNet"
})

# 3. Pivoteamos para tener una columna por Ataque (ACE/DLA) y Epsilon
df_cortex_final = df_raw.pivot(index='Epsilon', columns='Ataque', values=['ASR_Surr', 'ITF_LGBM', 'ITF_ResNet'])

# 4. Aplanamos los niveles de las columnas para visualización clara
df_cortex_final.columns = [f"{atk} {metrica.replace('_', ' ')}" for metrica, atk in df_cortex_final.columns]
df_cortex_final = df_cortex_final.reset_index()

# 5. Guardado único y definitivo a disco
joblib.dump(df_cortex_final, os.path.join(CHECKPOINT_DIR, "resultados_cortex_final.pkl"))
print(f"[✔] Resultados guardados exitosamente en {CHECKPOINT_DIR}/resultados_cortex_final.pkl")

# 6. Mostramos la tabla con un gradiente de color (Purples para diferenciarlo de SPECTRA)
columnas_ver = ['Epsilon',
                'ACE ASR Surr', 'ACE ITF LGBM', 'ACE ITF ResNet',
                'DLA (CORTEX) ASR Surr', 'DLA (CORTEX) ITF LGBM', 'DLA (CORTEX) ITF ResNet']

display(df_cortex_final[columnas_ver].style.background_gradient(
    cmap='Purples',
    subset=['ACE ITF ResNet', 'DLA (CORTEX) ITF ResNet']
))

# **Búsqueda del exploit óptimo por clase**
- Contribución original y solución realista para el TFG

# **PROPUESTA DE SIMULACIÓN DE ATAQUE APT SOFISTICADO PARA SPECTRA**

In [ ]:
"""
CORTEX FORENSICS: Búsqueda Dual de Escalada (ACE + DLA)
Simula a un atacante APT avanzado: busca la evasión absoluta (Caja Gris -> Caja Negra)
compitiendo entre la optimización directa de fronteras (ACE) y el secuestro latente (DLA).
El objetivo es encontrar el "Exploit Dorado" evaluando el Sigilo Compuesto (L2 + L0).
"""
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from src.attacks.ace import ACEAttack
from src.attacks.dla import DLAAttack

print("="*95)
print("CORTEX ADVANCED FORENSICS — Búsqueda Dual de Exploit Mínimo Viable (ACE vs DLA)")
print("="*95)

# Criterio de sigilo compuesto: penalizar tanto L2 como L0
def sigilo_score(l2, l0):
    """Menor es más sigiloso. Combina energía (L2) y dispersión (L0)."""
    return l2 * (1 + np.log1p(l0))

# 1. Parrilla de presupuestos de ataque (Epsilon sweep)
eps_grid = [0.01, 0.05, 0.10, 0.20, 0.35, 0.50, 1.0, 2.0]

# 2. Registro Maestro de Exploits (Añadimos 'score')
n_samples = len(X_surr_sc_valid)
best_exploits_global = {
    i: {'evaded': False, 'eps': np.inf, 'l2': np.inf, 'l0': np.inf, 'score': np.inf,
        'attack_used': 'None', 'X_adv_surr': None,
        'X_adv_vic_raw': None, 'X_adv_vic_sc': None}
    for i in range(n_samples)
}

print(f"[*] Iniciando explotación de sigilo sobre {n_samples} flujos válidos...")
print(f"[*] Estrategia: ACE (Fronteras) vs DLA (Anclas Latentes) | Steps: 10")
print(f"[*] Métrica de Victoria: Sigilo Compuesto (L2 ponderado por L0)")

# 3. BUCLE DE ESCALADA DUAL
for eps in tqdm(eps_grid, desc="🔥 CORTEX Dual Search", unit="eps", colour="magenta"):

    # --- A. MOTOR ACE (La Ganzúa de Frontera) ---
    atk_ace = ACEAttack(
        constraints=dc_surrogate, epsilon=eps, alpha=eps/3.0,
        steps=10, momentum=0.0, random_start=True,
        adaptive_alpha=False, loss_mode='targeted', device=device
    )
    res_ace = atk_ace.run(X_surr_sc_valid, y_valid, surrogate_wrapper)
    X_adv_ace_np = res_ace.X_adv

    X_ace_vic_raw, X_ace_vic_sc = expand_surrogate_to_victim(
        dc_surrogate.to_physical_space(X_adv_ace_np), X_vic_raw_valid
    )
    evaded_ace = (lgbm_victim.predict(X_ace_vic_raw) == 0) & (resnet_victim.predict(X_ace_vic_sc) == 0)
    l2_ace = np.linalg.norm(X_adv_ace_np - X_surr_sc_valid, axis=1)
    l0_ace = (np.abs(X_adv_ace_np - X_surr_sc_valid) > 1e-4).sum(axis=1)

    # --- B. MOTOR DLA (El Secuestro Latente) ---
    atk_dla = DLAAttack(
        constraints=dc_surrogate, target_layer=target_layer_dla, X_anchors=X_anchors_surr_sc,
        extraction_mode='grey_box', epsilon=eps, alpha=eps/3.0,
        steps=10, latent_weight=0.2, adaptive_alpha=False, device=device
    )
    res_dla = atk_dla.run(X_surr_sc_valid, y_valid, surrogate_wrapper)
    X_adv_dla_np = res_dla.X_adv if hasattr(res_dla, 'X_adv') else res_dla

    X_dla_vic_raw, X_dla_vic_sc = expand_surrogate_to_victim(
        dc_surrogate.to_physical_space(X_adv_dla_np), X_vic_raw_valid
    )
    evaded_dla = (lgbm_victim.predict(X_dla_vic_raw) == 0) & (resnet_victim.predict(X_dla_vic_sc) == 0)
    l2_dla = np.linalg.norm(X_adv_dla_np - X_surr_sc_valid, axis=1)
    l0_dla = (np.abs(X_adv_dla_np - X_surr_sc_valid) > 1e-4).sum(axis=1)

    # --- C. COMPETICIÓN DE SIGILO ---
    for i in range(n_samples):
        # Competición ACE
        score_ace = sigilo_score(l2_ace[i], l0_ace[i])
        if evaded_ace[i] and score_ace < best_exploits_global[i]['score']:
            best_exploits_global[i] = {
                'evaded': True, 'eps': eps, 'l2': l2_ace[i], 'l0': l0_ace[i], 'score': score_ace,
                'attack_used': 'ACE', 'X_adv_surr': X_adv_ace_np[i],
                'X_adv_vic_raw': X_ace_vic_raw[i], 'X_adv_vic_sc': X_ace_vic_sc[i]
            }
        # Competición DLA
        score_dla = sigilo_score(l2_dla[i], l0_dla[i])
        if evaded_dla[i] and score_dla < best_exploits_global[i]['score']:
            best_exploits_global[i] = {
                'evaded': True, 'eps': eps, 'l2': l2_dla[i], 'l0': l0_dla[i], 'score': score_dla,
                'attack_used': 'DLA', 'X_adv_surr': X_adv_dla_np[i],
                'X_adv_vic_raw': X_dla_vic_raw[i], 'X_adv_vic_sc': X_dla_vic_sc[i]
            }

    # --- D. EARLY STOPPING DEL GRID ---
    n_encontrados = sum(1 for v in best_exploits_global.values() if v['evaded'])
    cobertura = n_encontrados / n_samples
    if cobertura > 0.90:
        print(f"\n  [✓] CORTEX Early Stop — Cobertura crítica del {cobertura*100:.1f}% alcanzada en ε={eps:.2f}")
        break

# 4. EXTRACCIÓN DE RESULTADOS POR CLASE
print(f"\n{'─'*115}")
print(f"  {'Clase':<16} {'N válidas':>10} {'N evadidas':>10} {'ASR (%)':>8} "
      f"{'L2 mín':>8} {'L0 mín':>6} {'Score':>8}  {'Mejor Motor':<12} {'Eps':>5}")
print(f"{'─'*115}")

CLASS_NAMES_MAP = {
    1: 'DoS', 2: 'DDoS', 3: 'Web/Injection',
    4: 'Brute Force', 5: 'Recon', 6: 'Malware', 7: 'Exploits'
}

exploits_por_clase = {}
total_evadidas = sum(1 for v in best_exploits_global.values() if v['evaded'])

for cls_id, cls_name in CLASS_NAMES_MAP.items():
    idx_clase = np.where(y_valid == cls_id)[0]
    n_validas = len(idx_clase)
    if n_validas == 0: continue

    evadidos_clase = [i for i in idx_clase if best_exploits_global[i]['evaded']]
    n_evadidas = len(evadidos_clase)
    asr_cls = (n_evadidas / n_validas) * 100

    if n_evadidas > 0:
        # Ahora seleccionamos el Exploit Dorado basándonos en el SCORE de sigilo, no solo en L2
        mejor_idx = min(evadidos_clase, key=lambda x: best_exploits_global[x]['score'])
        mejor_data = best_exploits_global[mejor_idx]

        exploits_por_clase[cls_id] = {
            'cls_name'    : cls_name,
            'idx_global'  : mejor_idx,
            'X_orig_surr' : X_surr_sc_valid[mejor_idx],
            'X_adv_surr'  : mejor_data['X_adv_surr'],
            'X_orig_vic'  : X_vic_raw_valid[mejor_idx],
            'X_adv_vic'   : mejor_data['X_adv_vic_raw'],
            'l2_min'      : mejor_data['l2'],
            'l0_min'      : mejor_data['l0'],
            'score'       : mejor_data['score'],
            'eps_optimo'  : mejor_data['eps'],
            'asr_cls'     : asr_cls,
        }

        print(f"  {cls_name:<16} {n_validas:>10,} {n_evadidas:>10,} "
              f"{asr_cls:>8.1f}% {mejor_data['l2']:>8.4f} {mejor_data['l0']:>6} {mejor_data['score']:>8.2f}  "
              f"{mejor_data['attack_used']:<12} {mejor_data['eps']:>5.2f}")
    else:
        print(f"  {cls_name:<16} {n_validas:>10,} {'0':>10} "
              f"{'0.0%':>9} {'—':>8} {'—':>6} {'—':>8}  {'—':<12} {'—':>5}")

print(f"{'─'*115}")
print(f"  Resumen Global CORTEX: {total_evadidas}/{n_samples} flujos evadieron la Defensa Híbrida (ASR Total Dual: {(total_evadidas/n_samples)*100:.1f}%)")

# **Análisis semántico del exploit por clase**

In [ ]:
"""
CORTEX SEMANTICS — Informe de Inteligencia de Amenazas (CTI)
Extrae el "Exploit Dorado" (mejor Score L2+L0) de cada clase de ataque
y traduce la perturbación matemática a Tácticas, Técnicas y Procedimientos (TTPs).
"""
import numpy as np

print("="*105)
print(" 🛑 CORTEX THREAT INTELLIGENCE — Autopsia Táctica del Exploit Óptimo por Clase")
print("="*105)

CLASS_NAMES_MAP = {
    1: 'DoS', 2: 'DDoS', 3: 'Web/Injection',
    4: 'Brute Force', 5: 'Recon', 6: 'Malware', 7: 'Exploits'
}

# Diccionario táctico agrupado por Vectores de Evasión
TACTICAL_VECTORS = {
    'Temporal (Timing)': ['SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV', 'FLOW_DURATION_MILLISECONDS'],
    'Volumétrico (Payload)': ['IN_BYTES', 'IN_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT'],
    'Estructural (Paquetes)': ['LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
                               'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES'],
    'Topología/Protocolo': ['MIN_TTL', 'TCP_WIN_MAX_IN']
}

FEATURE_SEMANTICS = {
    'IN_BYTES'                   : 'Volumen de inyección de datos',
    'IN_PKTS'                    : 'Densidad de paquetes',
    'MIN_TTL'                    : 'Distancia de red aparente (Saltos)',
    'LONGEST_FLOW_PKT'           : 'Fragmentación del paquete mayor',
    'SHORTEST_FLOW_PKT'          : 'Relleno del paquete menor',
    'MIN_IP_PKT_LEN'             : 'Base de longitud IP',
    'MAX_IP_PKT_LEN'             : 'Techo de longitud IP',
    'SRC_TO_DST_AVG_THROUGHPUT'  : 'Ancho de banda consumido',
    'TCP_WIN_MAX_IN'             : 'Tamaño de ventana TCP (Congestión)',
    'SRC_TO_DST_IAT_AVG'         : 'Cadencia media de inyección',
    'SRC_TO_DST_IAT_STDDEV'      : 'Jitter/Aleatoriedad del ataque',
    'FLOW_DURATION_MILLISECONDS' : 'Ventana de exposición del flujo',
    'NUM_PKTS_UP_TO_128_BYTES'   : 'Micro-paquetes de control',
    'NUM_PKTS_512_TO_1024_BYTES' : 'Paquetes de tamaño medio',
    'NUM_PKTS_1024_TO_1514_BYTES': 'Paquetes de carga pesada',
}

exploits_dorados = 0

for cls_id, cls_name in CLASS_NAMES_MAP.items():
    idx_clase = np.where(y_valid == cls_id)[0]
    if len(idx_clase) == 0: continue

    evadidos_clase = [i for i in idx_clase if best_exploits_global[i]['evaded']]

    if len(evadidos_clase) > 0:
        exploits_dorados += 1
        mejor_idx = min(evadidos_clase, key=lambda x: best_exploits_global[x]['score'])
        exploit = best_exploits_global[mejor_idx]

        X_orig = X_surr_sc_valid[mejor_idx]
        X_adv  = exploit['X_adv_surr']

        # Proyección al espacio físico real
        X_orig_phys = dc_surrogate.to_physical_space(X_orig[np.newaxis])[0]
        X_adv_phys  = dc_surrogate.to_physical_space(X_adv[np.newaxis])[0]

        diff_mask = np.abs(X_adv - X_orig) > 1e-4
        n_l0 = diff_mask.sum()

        print(f"\n{'═'*105}")
        print(f" 🎯 TARGET: {cls_name:<15} | Evasión: {(len(evadidos_clase)/len(idx_clase))*100:.1f}% de las muestras")
        print(f" ⚙️  MOTOR : {exploit['attack_used']:<15} | Presupuesto (ε): {exploit['eps']:.2f}")
        print(f" 📊 HUELLA: Score={exploit['score']:.2f} | Energía (L2)={exploit['l2']:.3f} | Dispersión (L0)={n_l0} variables")
        print(f"{'═'*105}")

        cambios_por_vector = {k: [] for k in TACTICAL_VECTORS.keys()}

        for idx in np.where(diff_mask)[0]:
            if idx >= len(feature_names_surrogate): continue

            feat_name = feature_names_surrogate[idx]
            v_orig    = X_orig_phys[idx]
            v_adv     = X_adv_phys[idx]
            delta_abs = v_adv - v_orig

            if abs(v_orig) > 1e-4:
                delta_pct = (delta_abs / abs(v_orig)) * 100
                delta_str = f"{delta_pct:>+8.1f}%"
            else:
                delta_str = f"  +Abs({delta_abs:.1f})"

            vector_encontrado = "Topología/Protocolo"
            for vector, features in TACTICAL_VECTORS.items():
                if feat_name in features:
                    vector_encontrado = vector
                    break

            cambios_por_vector[vector_encontrado].append((feat_name, v_orig, v_adv, delta_str, delta_abs))

        # IMPRESIÓN DEL INFORME TÁCTICO
        for vector, cambios in cambios_por_vector.items():
            if len(cambios) > 0:
                print(f"\n 🛡️  VECTOR: {vector.upper()}")
                for feat_name, v_orig, v_adv, delta_str, delta_abs in cambios:
                    semantica = FEATURE_SEMANTICS.get(feat_name, feat_name)
                    accion = "▲ AUMENTAR" if delta_abs > 0 else "▼ REDUCIR "
                    # Etiqueta combinada: Contexto Semántico + Nombre de Feature Exacto
                    etiqueta = f"{semantica} [{feat_name}]"
                    print(f"    {accion} {etiqueta:<60} | {v_orig:>10.2f} → {v_adv:>10.2f} [{delta_str}]")

    else:
        print(f"\n 🛡️  DEFENSA EXITOSA: {cls_name:<15} | La arquitectura repelió todos los ataques.")

print(f"\n{'═'*105}")
print(f"[✓] Análisis Forense Completado. Se aislaron {exploits_dorados} Exploits Dorados de {len(CLASS_NAMES_MAP)} clases posibles.")

## HIPÓTESIS Y JUSTIFICACIÓN

Para evadir la detección de DoS, el ataque recomienda reducir los paquetes y bytes en un 70%. Pero es cierto que un DoS se detecta por volumen. Si el atacante reduce el volumen, el NIDS no lo ve... pero deja de ser un DoS efectivo.

En cambio, para el Malware, el ataque recomienda aumentar la velocidad y tamaño. Como el malware suele ser un tráfico "goteo" (C2 beacons), inyectar ruido o hacerlo un poco más rápido descoloca al modelo sin romper el payload malicioso.

# **Gráficas de transferibilidad y sigilo**

In [ ]:
"""
================================================================================
CORTEX VISUALIZATION: Transferibilidad Grey-Box y Eficiencia de Sigilo
================================================================================
"""
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# 1. Preparación de datos desde la memoria de la ejecución
df_cortex_plot = pd.DataFrame(resultados_cortex)
df_ace = df_cortex_plot[df_cortex_plot['Ataque'] == 'ACE'].reset_index(drop=True)
df_dla = df_cortex_plot[df_cortex_plot['Ataque'].str.contains('DLA')].reset_index(drop=True)
epsilons_plot = df_ace['Epsilon'].values

# Calculamos el ITF de LightGBM (ya teníamos el de ResNet en la tabla)
itf_lgbm_ace = np.where(df_ace['ASR Surrogate (%)'] > 0.5, df_ace['ASR Vic LightGBM (%)'] / df_ace['ASR Surrogate (%)'], 0)
itf_lgbm_dla = np.where(df_dla['ASR Surrogate (%)'] > 0.5, df_dla['ASR Vic LightGBM (%)'] / df_dla['ASR Surrogate (%)'], 0)

# Paleta CORTEX (Azules para ACE, Morados para DLA)
colores = {
    'ACE_surr': '#74b9ff', 'ACE_vic': '#0984e3',
    'DLA_surr': '#a29bfe', 'DLA_vic': '#6c5ce7',
}

# 2. Configuración del Canvas
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

# ── Panel 1: ASR sobre Surrogate (Gemelo Digital) ─────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epsilons_plot, df_ace['ASR Surrogate (%)'], 'o--', color=colores['ACE_surr'], lw=2, ms=7, label='ACE Surrogate')
ax1.plot(epsilons_plot, df_dla['ASR Surrogate (%)'], 's-',  color=colores['DLA_surr'], lw=2, ms=7, label='DLA Surrogate')
ax1.set_title('ASR sobre Digital Twin\n(Espacio 30-Features)', fontweight='bold')
ax1.set_xlabel('Epsilon (Presupuesto)'); ax1.set_ylabel('ASR (%)')
ax1.set_xticks(epsilons_plot); ax1.grid(True, ls=':', alpha=0.6)
ax1.legend(fontsize=10)

# ── Panel 2: ASR transferido a Víctima (LightGBM) ─────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epsilons_plot, df_ace['ASR Vic LightGBM (%)'], 'o--', color=colores['ACE_vic'], lw=2, ms=7, label='ACE → LightGBM')
ax2.plot(epsilons_plot, df_dla['ASR Vic LightGBM (%)'], 's-',  color=colores['DLA_vic'], lw=2, ms=7, label='DLA → LightGBM')
ax2.set_title('ASR Transferido\nLightGBM Víctima (66-Features)', fontweight='bold')
ax2.set_xlabel('Epsilon'); ax2.set_ylabel('ASR (%)')
ax2.set_xticks(epsilons_plot); ax2.grid(True, ls=':', alpha=0.6)
ax2.legend(fontsize=10)

# ── Panel 3: ASR transferido a Víctima (ResNet) ───────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(epsilons_plot, df_ace['ASR Vic ResNet (%)'], 'o--', color=colores['ACE_vic'], lw=2, ms=7, label='ACE → ResNet')
ax3.plot(epsilons_plot, df_dla['ASR Vic ResNet (%)'], 's-',  color=colores['DLA_vic'], lw=2, ms=7, label='DLA → ResNet')
ax3.set_title('ASR Transferido\nResNet Víctima (66-Features)', fontweight='bold')
ax3.set_xlabel('Epsilon'); ax3.set_ylabel('ASR (%)')
ax3.set_xticks(epsilons_plot); ax3.grid(True, ls=':', alpha=0.6)
ax3.legend(fontsize=10)

# ── Panel 4: ITF (Índice de Transferibilidad) hacia LightGBM ──────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(epsilons_plot, itf_lgbm_ace, 'o--', color=colores['ACE_vic'], lw=2, ms=7, label='ACE ITF LightGBM')
ax4.plot(epsilons_plot, itf_lgbm_dla, 's-',  color=colores['DLA_vic'], lw=2, ms=7, label='DLA ITF LightGBM')
ax4.axhline(0.4, color='red', ls='--', lw=1.5, alpha=0.7, label='Umbral crítico (0.4)')
ax4.set_title('Índice de Transferencia (ITF)\nhacia LightGBM', fontweight='bold')
ax4.set_xlabel('Epsilon'); ax4.set_ylabel('ITF')
ax4.set_xticks(epsilons_plot); ax4.grid(True, ls=':', alpha=0.6)
ax4.set_ylim(0, 0.6); ax4.legend(fontsize=9)

# ── Panel 5: ITF (Índice de Transferibilidad) hacia ResNet ────────────────
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(epsilons_plot, df_ace['ITF ResNet'], 'o--', color=colores['ACE_vic'], lw=2, ms=7, label='ACE ITF ResNet')
ax5.plot(epsilons_plot, df_dla['ITF ResNet'], 's-',  color=colores['DLA_vic'], lw=2, ms=7, label='DLA ITF ResNet')
ax5.axhline(0.2, color='red', ls='--', lw=1.5, alpha=0.7, label='Umbral crítico (0.2)')
ax5.set_title('Índice de Transferencia (ITF)\nhacia ResNet', fontweight='bold')
ax5.set_xlabel('Epsilon'); ax5.set_ylabel('ITF')
ax5.set_xticks(epsilons_plot); ax5.grid(True, ls=':', alpha=0.6)
ax5.set_ylim(0, 0.4); ax5.legend(fontsize=9)

# ── Panel 6: Heatmap del Exploit Dorado por Clase (Métrica SCORE) ─────────
ax6 = fig.add_subplot(gs[1, 2])
clases_con_exploit = list(exploits_por_clase.keys())
asr_por_clase = [exploits_por_clase[c]['asr_cls'] for c in clases_con_exploit]
nombres_clase = [exploits_por_clase[c]['cls_name'] for c in clases_con_exploit]
score_por_clase = [exploits_por_clase[c]['score'] for c in clases_con_exploit]
l0_por_clase = [exploits_por_clase[c]['l0_min'] for c in clases_con_exploit]

# Usamos la paleta morada (Purples) para mantener el theme de CORTEX
bars = ax6.barh(nombres_clase, asr_por_clase,
                color=plt.cm.Purples([min(1.0, (a+10)/100) for a in asr_por_clase]),
                edgecolor='black', linewidth=0.5)

for bar, score, l0 in zip(bars, score_por_clase, l0_por_clase):
    ax6.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'Sc={score:.1f} | L0={l0}', va='center', fontsize=9)

ax6.set_title('Evasión por Clase de Ataque\n(Basado en Sigilo Compuesto)', fontweight='bold')
ax6.set_xlabel('ASR Dual (%)'); ax6.grid(True, ls=':', alpha=0.6, axis='x')
ax6.set_xlim(0, max(asr_por_clase) + 15 if asr_por_clase else 100)

# ── Título Global y Guardado ──────────────────────────────────────────────
fig.suptitle(
    'CORTEX — Transferibilidad Grey-Box: ACE vs DLA sobre Surrogate NF-UQ-NIDS-V2\n'
    'Evaluación de inyección de ruido de frontera vs. secuestro del espacio latente.',
    fontsize=15, fontweight='bold', y=1.02, color='#2c3e50'
)

CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'resultados_cortex')
plt.savefig(
    os.path.join(CHECKPOINT_DIR, 'cortex_transferibilidad.png'),
    dpi=300, bbox_inches='tight'
)
plt.show()
print(f"[✔] Gráfica guardada en: {CHECKPOINT_DIR}/cortex_transferibilidad.png")

## 5. Análisis del rol defensivo del IP Behavior Buffer

Si el ITF es bajo, el buffer es la razón principal.
Demostramos esto comparando la predicción del víctima con y sin las features BUF_*.

In [ ]:
"""
ESTUDIO DE ABLACIÓN DEL IP BEHAVIOR BUFFER (CORTEX)
Comparamos el daño de CORTEX (ACE eps=2.0) contra el IDS real
y contra un IDS "ciego" al que le borramos el historial temporal de la IP.
"""
import numpy as np
import pandas as pd
from src.attacks.ace import ACEAttack

print("="*85)
print("DEMOSTRACIÓN DE DEFENSA ACTIVA — Ablación del IP Buffer (CORTEX)")
print("="*85)

# 1. Generamos el ataque de máxima potencia (ACE eps=2.0)
eps_max = 2.0
atk_ablation = ACEAttack(
    constraints=dc_surrogate,
    epsilon=eps_max,
    alpha=eps_max/3.0,
    steps=10,
    momentum=0.0,
    random_start=True,
    adaptive_alpha=False,
    device=device
)

print(f"[-] Generando ataque ACE masivo (eps={eps_max}) sobre el Digital Twin...")
res_abl = atk_ablation.run(X_surr_sc_valid, y_valid, surrogate_wrapper)
X_adv_abl_np = res_abl.X_adv if hasattr(res_abl, 'X_adv') else res_abl

# Calculamos el ASR en el Surrogate para la gráfica
asr_surr_abl = (surrogate_wrapper.predict(X_adv_abl_np) == 0).mean() * 100

# 2. Expandimos al espacio físico de la víctima
X_adv_vic_raw_original, X_adv_vic_sc_original = expand_surrogate_to_victim(
    dc_surrogate.to_physical_space(X_adv_abl_np), X_vic_raw_valid
)

# 3. CREAMOS LA REALIDAD ALTERNATIVA (Sin IP Buffer)
print("[-] Borrando el historial del IP Buffer (Simulando IDS sin estado temporal)...")
buf_mask = np.array(['BUF_' in name for name in dc.feature_names])
buf_indices = np.where(buf_mask)[0]

X_adv_vic_raw_no_buf = X_adv_vic_raw_original.copy()
# Ponemos a 0 el historial (como si fuera el primer paquete de una IP desconocida)
X_adv_vic_raw_no_buf[:, buf_indices] = 0.0

# Escalamos esta nueva realidad para la ResNet
X_adv_vic_sc_no_buf = dc.to_scaled_space(X_adv_vic_raw_no_buf)

# 4. PREDICCIONES CARA A CARA
print("[-] Evaluando impacto...\n")

# --- CON BUFFER (Realidad) ---
y_pred_lgbm_con = lgbm_victim.predict(X_adv_vic_raw_original)
y_pred_resnet_con = resnet_victim.predict(X_adv_vic_sc_original)
evasion_lgbm_con = (y_pred_lgbm_con == 0).mean() * 100
evasion_resnet_con = (y_pred_resnet_con == 0).mean() * 100

# --- SIN BUFFER (Ablación) ---
y_pred_lgbm_sin = lgbm_victim.predict(X_adv_vic_raw_no_buf)
y_pred_resnet_sin = resnet_victim.predict(X_adv_vic_sc_no_buf)
evasion_lgbm_sin = (y_pred_lgbm_sin == 0).mean() * 100
evasion_resnet_sin = (y_pred_resnet_sin == 0).mean() * 100

# 5. INFORME DE DEFENSA
print(f"{'─'*75}")
print(f"  {'Escenario':<25} {'Evasión LightGBM':>20} {'Evasión ResNet':>20}")
print(f"{'─'*75}")
print(f"  A) Con IP Buffer (Real)    {evasion_lgbm_con:>19.1f}% {evasion_resnet_con:>19.1f}%")
print(f"  B) Sin IP Buffer (Ciego)   {evasion_lgbm_sin:>19.1f}% {evasion_resnet_sin:>19.1f}%")
print(f"{'─'*75}")

delta_lgbm = evasion_lgbm_sin - evasion_lgbm_con
delta_resnet = evasion_resnet_sin - evasion_resnet_con

print(f"  [!] ATAQUES BLOQUEADOS POR EL BUFFER (Delta):")
print(f"      → LightGBM : Salvó al sistema de un {delta_lgbm:+.1f}% de ataques.")
print(f"      → ResNet   : Salvó al sistema de un {delta_resnet:+.1f}% de ataques.")

-------------------------

## 6. Visualización de resultados

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Configuramos colores corporativos/académicos (Paleta CORTEX)
COLOR_SURR    = '#74b9ff'  # Azul claro (Digital Twin)
COLOR_LGBM    = '#0984e3'  # Azul fuerte (Víctima LightGBM)
COLOR_RES     = '#6c5ce7'  # Morado (Víctima ResNet)
COLOR_SIN_BUF = '#b2bec3'  # Gris para el "Sin Buffer"

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ==============================================================================
# Panel 1 — El Espejismo del Atacante (ASR sobre Surrogate vs Realidad)
# ==============================================================================
modelos = ['Digital Twin\n(ACE eps=2.0)', 'Víctima\n(LightGBM)', 'Víctima\n(ResNet)']
asrs    = [asr_surr_abl, evasion_lgbm_con, evasion_resnet_con]
colores = [COLOR_SURR, COLOR_LGBM, COLOR_RES]

axes[0].bar(modelos, asrs, color=colores, width=0.5, edgecolor='black', linewidth=1)
axes[0].set_ylabel('Evasión (ASR %)', fontweight='bold')
axes[0].set_title('El Espejismo del Atacante\n(Ataque Caja Gris Neural)', fontweight='bold', pad=15)
axes[0].set_ylim(0, max(asrs) * 1.2 + 5)
for i, v in enumerate(asrs):
    axes[0].text(i, v + 1.5, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=11)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# ==============================================================================
# Panel 2 — Rol Defensivo del IP Buffer (Ablación Global)
# ==============================================================================
categorias = ['Con IP Buffer\n(Tu Arquitectura)', 'Sin IP Buffer\n(NIDS Tradicional)']
vals_resnet = [evasion_resnet_con, evasion_resnet_sin]
vals_lgbm   = [evasion_lgbm_con, evasion_lgbm_sin]

x = np.arange(len(categorias))
w = 0.3
axes[1].bar(x - w/2, vals_resnet, w, label='ResNet', color=COLOR_RES, edgecolor='black', linewidth=1)
axes[1].bar(x + w/2, vals_lgbm,   w, label='LightGBM', color=COLOR_LGBM, edgecolor='black', linewidth=1)
axes[1].set_ylabel('Ataques que lograron evadir (%)', fontweight='bold')
axes[1].set_title('Estudio de Ablación:\nEl IP Buffer como Escudo Activo', fontweight='bold', pad=15)
axes[1].set_xticks(x)
axes[1].set_xticklabels(categorias, fontweight='bold')

# Anotaciones de la degradación
axes[1].annotate(f'Δ +{evasion_resnet_sin - evasion_resnet_con:.1f}%', xy=(1 - w/2, evasion_resnet_sin + 2), ha='center', color=COLOR_RES, fontweight='bold')
axes[1].annotate(f'Δ +{evasion_lgbm_sin - evasion_lgbm_con:.1f}%', xy=(1 + w/2, evasion_lgbm_sin + 2), ha='center', color=COLOR_LGBM, fontweight='bold')

axes[1].legend()
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

# ==============================================================================
# Panel 3 — Colapso del NIDS por Clases (Enfocado en ResNet para CORTEX)
# ==============================================================================
clases_labels = []
ev_con_list = []
ev_sin_list = []

for cls_id, cls_name in CLASS_NAMES_MAP.items():
    idx_clase = np.where(y_valid == cls_id)[0]
    if len(idx_clase) == 0: continue

    clases_labels.append(cls_name)
    ev_con_list.append((y_pred_resnet_con[idx_clase] == 0).mean() * 100)
    ev_sin_list.append((y_pred_resnet_sin[idx_clase] == 0).mean() * 100)

y_pos = np.arange(len(clases_labels))
h = 0.35

# Invertimos el orden para que se lea mejor de arriba a abajo
axes[2].barh(y_pos + h/2, ev_sin_list, h, label='Sin IP Buffer (Ciego)', color=COLOR_SIN_BUF, edgecolor='black', linewidth=1)
axes[2].barh(y_pos - h/2, ev_con_list, h, label='Con IP Buffer (Real)', color=COLOR_RES, edgecolor='black', linewidth=1)

axes[2].set_yticks(y_pos)
axes[2].set_yticklabels(clases_labels, fontsize=10, fontweight='bold')
axes[2].set_xlabel('Evasión Efectiva (%)', fontweight='bold')
axes[2].set_title('Colapso de Seguridad por Tipo de Ataque\n(Víctima: TabularResNet)', fontweight='bold', pad=15)
axes[2].legend()
axes[2].grid(axis='x', linestyle='--', alpha=0.5)
axes[2].set_xlim(0, 105)

# Título General
plt.suptitle('CORTEX — Evaluación de Robustez y Defensa Activa (IP Behavior Buffer)',
             fontsize=16, fontweight='bold', y=1.05, color='#2c3e50')
plt.tight_layout()

# Guardar figura
plt.savefig(os.path.join(CHECKPOINT_DIR, 'cortex_ablacion_defensa.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"[✓] Figura de Ablación guardada en {CHECKPOINT_DIR}")